# Work Item Orchestrator — user perspective

**What it does.** Drives one Linear task through its full lifecycle in **one** `ClaudeSDKClient` session:

1. **Step 1 — Intelligence enrichment** (skill 10, INTL-01) — Glob/Grep over `knowledge/`, populates `knowledge_context`.
2. **Steps 2-4 — Builder → Git → QA cycle** (Phase 3, max 3 QA cycles).
3. **Step 5 — Knowledge storage evaluation** (skill 11, INTL-02) — writes new `knowledge/<cat>/*.md` entries when ingestion criteria are met.

**Architecture rule.** ONE SDK session. Phase 2 agents are exposed as in-process MCP tools via `create_sdk_mcp_server` + `@tool`. **No** sub-agent dispatch — `Task` is forbidden in the allow-list (G3).

**Codex hard-block.** WIO is Claude-only. `ClaudeSDKClient` has no Codex equivalent. `resolve_runtime("wio")` raises if `HSB_RUNTIME_WIO=codex`.

**Entry point.** `await run_orchestration_cycle(work_item_id)` — pass `None` to let it pick the lowest-id ready task.

**Cost.** Live runs are the heaviest single notebook in the suite. Gated.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Pure-logic probe — `assemble_system_prompt`

Concatenates the 7 SKILL.md files in injection order with `# SKILL: <stem>` separators. Always runs — no LLM, no Linear.

In [ ]:
import os

from hsb.agents.work_item_orchestrator import SKILL_FILES, assemble_system_prompt

# assemble_system_prompt() reads SKILL.md files relative to the current
# working directory. When this notebook is run from notebooks/agents/, cwd
# would not resolve `skills/...` — temporarily chdir to ROOT for the probe.
_prev_cwd = Path.cwd()
os.chdir(ROOT)
try:
    prompt = assemble_system_prompt()
finally:
    os.chdir(_prev_cwd)

print(f"system prompt length: {len(prompt):,} chars")
print(f"skill files: {len(SKILL_FILES)}")
for path in SKILL_FILES:
    marker = f"# SKILL: {Path(path).stem}"
    assert marker in prompt, f"missing separator: {marker}"
print("all skill separators present")

## Pure-logic probe — registered MCP tools

The WIO exposes Phase 2 agents as 4 in-process MCP tools (`run_linear_op`, `run_builder`, `run_git`, `run_qa`). Verify their decorator metadata is registered without instantiating the SDK session.

In [ ]:
from hsb.agents import work_item_orchestrator as wio_mod

names = []
for attr in ("run_linear_tool", "run_builder_tool", "run_git_tool", "run_qa_tool"):
    fn = getattr(wio_mod, attr)
    names.append(attr)
print("registered @tool wrappers:", names)
assert len(names) == 4, (
    "WIO must expose exactly 4 @tool wrappers (Linear/Builder/Git/QA)"
)
print("WIO tool registry OK")

## Live invocation (gated, expensive, Claude-only)

Runs the full lifecycle for one task. Set:

- `HSB_NOTEBOOK_RUN_LIVE=1`
- `HSB_NOTEBOOK_WIO_TASK_ID=LIN-...` (a sandbox Linear task)
- `CLAUDE_CODE_OAUTH_TOKEN` (G1: never `ANTHROPIC_API_KEY`)

The runtime is hard-blocked from Codex — `HSB_RUNTIME_WIO=codex` will raise.

In [ ]:
import os

from _helpers import assert_g1_safe, gated, live_mode, selected_runtime

task_id = os.environ.get("HSB_NOTEBOOK_WIO_TASK_ID")
if not live_mode() or not task_id:
    print(gated("WIO live run (set HSB_NOTEBOOK_WIO_TASK_ID=LIN-...)"))
elif selected_runtime("wio") == "codex":
    print("[skipped] HSB_RUNTIME_WIO=codex is hard-blocked — WIO is Claude-only")
else:
    assert_g1_safe()
    import asyncio

    from hsb.agents.work_item_orchestrator import run_orchestration_cycle

    asyncio.run(run_orchestration_cycle(task_id))
    print("WIO cycle complete for", task_id)